In [5]:
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import CountVectorizer

from gensim.models import Word2Vec, FastText

In [6]:
df = pd.read_csv("../data/cleaned_imdb_dataset.csv")

print("Dataset Loaded Successfully!")
print("Dataset Shape:", df.shape)

print(df.head())
print(df.columns)

Dataset Loaded Successfully!
Dataset Shape: (50000, 2)
                                      cleaned_review sentiment
0  one reviewers mention watch 1 oz episode hook ...  positive
1  wonderful little production film technique una...  positive
2  think wonderful way spend time hot summer week...  positive
3  basically there's family little boy jake think...  negative
4  petter mattei's love time money visually stun ...  positive
Index(['cleaned_review', 'sentiment'], dtype='str')


In [7]:
columns = {col.lower(): col for col in df.columns}

possible_columns = ["cleaned_review", "cleaned_text", "review", "text"]

text_column = None

for col in possible_columns:
    if col in columns:
        text_column = columns[col]
        break

if text_column is None:
    raise ValueError("Text column not found. Check the dataset columns.")

texts = df[text_column].dropna().astype(str)

texts = texts[texts.str.strip() != ""]

print("Total Documents:", len(texts))
print("Sample Review:")
print(texts.iloc[0])

Total Documents: 50000
Sample Review:
one reviewers mention watch 1 oz episode hook right exactly happen first thing strike oz brutality unflinching scenes violence set right word go trust not show faint hearted timid show pull no punch regard drug sex violence hardcore classic use word call oz nickname give oswald maximum security state penitentary focus mainly emerald city experimental section prison cells glass front face inwards privacy not high agenda em city home many aryans muslims gangstas latinos christians italians irish scuffle death star dodgy dealings shady agreements never far away would say main appeal show due fact go show wouldn't dare forget pretty picture paint mainstream audiences forget charm forget romance oz doesn't mess around first episode ever saw strike nasty surreal couldn't say ready watch develop taste oz get accustom high level graphic violence not violence injustice crook guard who'll sell nickel inmates who'll kill order get away well mannered middle cl

In [8]:
bow_vectorizer = CountVectorizer()

bow_matrix = bow_vectorizer.fit_transform(texts)

print("Bag of Words Matrix Shape:", bow_matrix.shape)

total_elements = bow_matrix.shape[0] * bow_matrix.shape[1]

zero_elements = total_elements - bow_matrix.nnz

sparsity = (zero_elements / total_elements) * 100

print("Sparsity:", sparsity, "%")

print("\nVocabulary Size:", len(bow_vectorizer.vocabulary_))

Bag of Words Matrix Shape: (50000, 93350)
Sparsity: 99.89611648634173 %

Vocabulary Size: 93350


In [9]:
print("First 20 Vocabulary Words:")

print(bow_vectorizer.get_feature_names_out()[:20])

print("\nFirst 5 Rows of BoW Matrix:")

print(bow_matrix[:5].toarray())

First 20 Vocabulary Words:
['00' '000' '00000000000' '0000000000001' '00000001' '00001' '00015'
 '000dm' '000s' '001' '003830' '006' '0069' '007' '0079' '007s' '0080'
 '0083' '009' '00am']

First 5 Rows of BoW Matrix:
[[0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]]


In [10]:
unigram_vectorizer = CountVectorizer(ngram_range=(1, 1))

unigram_matrix = unigram_vectorizer.fit_transform(texts)


bigram_vectorizer = CountVectorizer(ngram_range=(2, 2))

bigram_matrix = bigram_vectorizer.fit_transform(texts)


trigram_vectorizer = CountVectorizer(ngram_range=(3, 3))

trigram_matrix = trigram_vectorizer.fit_transform(texts)


print("Unigram Shape:", unigram_matrix.shape)

print("Bigram Shape:", bigram_matrix.shape)

print("Trigram Shape:", trigram_matrix.shape)

Unigram Shape: (50000, 93350)
Bigram Shape: (50000, 2778554)
Trigram Shape: (50000, 5437586)


In [11]:
def calculate_sparsity(matrix):

    total = matrix.shape[0] * matrix.shape[1]

    zeros = total - matrix.nnz

    return (zeros / total) * 100


ngram_results = pd.DataFrame({

    "N-gram": ["Unigram", "Bigram", "Trigram"],

    "Vocabulary Size": [
        unigram_matrix.shape[1],
        bigram_matrix.shape[1],
        trigram_matrix.shape[1]
    ],

    "Sparsity (%)": [
        calculate_sparsity(unigram_matrix),
        calculate_sparsity(bigram_matrix),
        calculate_sparsity(trigram_matrix)
    ]

})

print(ngram_results)

    N-gram  Vocabulary Size  Sparsity (%)
0  Unigram            93350     99.896116
1   Bigram          2778554     99.995716
2  Trigram          5437586     99.997792


In [12]:
print("Unigram Examples:")

print(unigram_vectorizer.get_feature_names_out()[:20])


print("\nBigram Examples:")

print(bigram_vectorizer.get_feature_names_out()[:20])


print("\nTrigram Examples:")

print(trigram_vectorizer.get_feature_names_out()[:20])

Unigram Examples:
['00' '000' '00000000000' '0000000000001' '00000001' '00001' '00015'
 '000dm' '000s' '001' '003830' '006' '0069' '007' '0079' '007s' '0080'
 '0083' '009' '00am']

Bigram Examples:
['00 01' '00 10' '00 19' '00 80' '00 84' '00 90' '00 acorn' '00 agent'
 '00 air' '00 alison' '00 back' '00 bargain' '00 believable' '00 bin'
 '00 blockbuster' '00 brief' '00 budget' '00 buy' '00 case' '00 cash']

Trigram Examples:
['00 01 percent' '00 10 000' '00 19 end' '00 80 irreplaceable'
 '00 84 minute' '00 90 minutes' '00 acorn media' '00 agent difficult'
 '00 agent level' '00 air horror' '00 alison agree' '00 back would'
 '00 bargain eddie' '00 believable set' '00 bin decide' '00 bin little'
 '00 bin wal' '00 blockbuster one' '00 brief review' '00 budget get']


In [13]:
tokenized_reviews = []

for review in texts:

    words = review.split()

    if len(words) > 0:

        tokenized_reviews.append(words)


print("Total Tokenized Reviews:", len(tokenized_reviews))

print("\nSample Tokenized Review:")

print(tokenized_reviews[0])

Total Tokenized Reviews: 50000

Sample Tokenized Review:
['one', 'reviewers', 'mention', 'watch', '1', 'oz', 'episode', 'hook', 'right', 'exactly', 'happen', 'first', 'thing', 'strike', 'oz', 'brutality', 'unflinching', 'scenes', 'violence', 'set', 'right', 'word', 'go', 'trust', 'not', 'show', 'faint', 'hearted', 'timid', 'show', 'pull', 'no', 'punch', 'regard', 'drug', 'sex', 'violence', 'hardcore', 'classic', 'use', 'word', 'call', 'oz', 'nickname', 'give', 'oswald', 'maximum', 'security', 'state', 'penitentary', 'focus', 'mainly', 'emerald', 'city', 'experimental', 'section', 'prison', 'cells', 'glass', 'front', 'face', 'inwards', 'privacy', 'not', 'high', 'agenda', 'em', 'city', 'home', 'many', 'aryans', 'muslims', 'gangstas', 'latinos', 'christians', 'italians', 'irish', 'scuffle', 'death', 'star', 'dodgy', 'dealings', 'shady', 'agreements', 'never', 'far', 'away', 'would', 'say', 'main', 'appeal', 'show', 'due', 'fact', 'go', 'show', "wouldn't", 'dare', 'forget', 'pretty', 'pict

In [24]:
cbow_model = Word2Vec(

    sentences=tokenized_reviews,

    vector_size=100,

    window=5,

    min_count=2,

    workers=4,

    sg=0,

    epochs=10,

    seed=42

)

print("Word2Vec CBOW Model Trained Successfully!")

import os

os.makedirs("../models", exist_ok=True)

cbow_model.save("../models/cbow_model.model")

Word2Vec CBOW Model Trained Successfully!


In [15]:
print("Vector Size:", cbow_model.vector_size)

print("Window Size:", cbow_model.window)

print("Minimum Count:", cbow_model.min_count)

print("Epochs:", cbow_model.epochs)

print("Vocabulary Size:", len(cbow_model.wv))

Vector Size: 100
Window Size: 5
Minimum Count: 2
Epochs: 10
Vocabulary Size: 66887


In [16]:
word = "movie"

if word in cbow_model.wv:

    print("Vector Representation of:", word)

    print(cbow_model.wv[word])

else:

    print(word, "is not present in the vocabulary.")

Vector Representation of: movie
[-1.320098    2.9903636   2.588298   -0.61396307  0.5312344   0.74482685
 -0.73989487 -0.51460695 -2.4716156  -0.25512108  2.1600096   0.44196308
 -1.3190535  -0.7661995  -2.13145     0.7350785  -0.6258707   0.6807919
 -0.12533675  2.1107693  -0.17653792  0.05881615 -0.9330626  -0.7832783
  2.716726    0.29364964 -0.7102199  -3.1999574  -1.0511719   1.5449079
 -0.51699966  1.1233919   0.68033886  0.3436504   0.62173027  2.8973854
  0.23063725 -0.30940232 -0.37913904 -0.04084839  0.48984867 -0.21753795
 -1.0146173   0.78769004 -1.4978666  -0.46444416 -1.919927    0.9811533
 -1.6601356  -1.2946067   0.4070145   4.0809326   1.1447475  -0.41444442
 -0.6728103  -0.44566265  0.40966064  0.8450133   0.54571325 -0.41470283
  0.34239382  0.88754374 -1.0183785  -0.2173803   0.6877383   0.06485662
  2.0119681  -1.6792827  -0.693822   -0.59243226 -0.15205272 -0.19958241
  2.0953164   0.5373035  -0.6482591  -0.68779606  0.18541467  0.3542523
  0.49494365 -0.10442975 

In [ ]:
fasttext_model = FastText(

    sentences=tokenized_reviews,

    vector_size=100,

    window=5,

    min_count=2,

    workers=4,

    sg=0,

    epochs=10,

    min_n=3,

    max_n=6,

    seed=42

)

print("fastText Model Trained Successfully!")

import os

os.makedirs("../models", exist_ok=True)

fasttext_model.save("../models/fasttext_model.model")

print("FastText model saved successfully!")

fastText Model Trained Successfully!


In [19]:
print("Vector Size:", fasttext_model.vector_size)

print("Window Size:", fasttext_model.window)

print("Minimum Count:", fasttext_model.min_count)

print("Epochs:", fasttext_model.epochs)

print("Vocabulary Size:", len(fasttext_model.wv))

Vector Size: 100
Window Size: 5
Minimum Count: 2
Epochs: 10
Vocabulary Size: 66887


In [20]:
known_word = "amazing"

if known_word not in fasttext_model.wv:

    known_word = fasttext_model.wv.index_to_key[0]


misspelled_word = known_word[:-1]

print("Original Word:", known_word)

print("Misspelled Word:", misspelled_word)

Original Word: amazing
Misspelled Word: amazin


In [21]:
try:

    print(cbow_model.wv[misspelled_word])

except KeyError:

    print("Word2Vec cannot generate a vector for this OOV word.")

Word2Vec cannot generate a vector for this OOV word.


In [22]:
fasttext_vector = fasttext_model.wv.get_vector(misspelled_word)

print("fastText generated a vector successfully!")

print("Vector:")

print(fasttext_vector)

fastText generated a vector successfully!
Vector:
[ 0.7652591   0.44641823 -0.4884459   1.5163254   1.1694132   0.2834426
  0.5085776  -0.14887665  0.92534053  0.02731375  0.35327947 -0.43450224
  0.54680645  2.5881202  -1.1502404  -0.5730412  -0.13287322  0.37581974
 -0.50813544  0.06493277 -0.01718913 -0.57261395 -0.5258119  -0.38297313
  0.5869891  -0.58022976  0.4799298  -1.0473211   1.5765789  -1.3563325
  0.02787296  0.43021333  0.37292084  1.2851073   1.0830209  -0.47706223
 -1.2768246  -0.17875111  0.0407651   0.53142613 -0.07617129 -0.6861628
  0.23830755  0.10027109  1.0275317   0.2362238   0.24571487 -1.1871825
  0.06982649  0.7287542   0.98994267  0.35128507 -0.6449634  -0.37391424
 -0.74906325  0.5294778  -0.9284265   0.57346565 -0.49464518 -0.1930721
  0.29738486 -0.37465942 -0.16049941 -0.00850666  0.5607445  -0.16988005
 -0.46052134 -0.02282794  0.22805507 -0.1601564   0.80259967  0.11403709
  1.0540719  -0.29603955  0.060377    0.4386492   0.406638    0.03716802
  0.74

In [23]:
similar_words = fasttext_model.wv.most_similar(

    positive=[fasttext_vector],

    topn=5

)

print("Similar Words:")

for word, score in similar_words:

    print(word, ":", score)

Similar Words:
amazing! : 0.780880868434906
amazing!! : 0.7740241289138794
amazed! : 0.7508761882781982
amaze : 0.7246084213256836
amazing!!! : 0.6860747337341309
